# Handwritten Mathematical Expression Recognition - Training Workflow

Chào mừng đến với notebook huấn luyện mô hình nhận dạng biểu thức toán học viết tay.

Notebook này tổng hợp toàn bộ quy trình làm việc vào một nơi duy nhất, cho phép bạn thực thi từng bước một cách tuần tự:
1.  **Cài đặt môi trường**: Cài đặt các thư viện cần thiết.
2.  **Cấu hình**: Định nghĩa tất cả các tham số cho mô hình và quá trình huấn luyện.
3.  **Tiền xử lý dữ liệu**: Chuẩn hóa và chuẩn bị dữ liệu hình ảnh.
4.  **Định nghĩa Dataset & Model**: Nạp các lớp Dataset và kiến trúc Vision Transformer.
5.  **Huấn luyện**: Chạy vòng lặp huấn luyện chính.

---
**Lưu ý**: Hãy đảm bảo rằng bạn đã tải xuống và giải nén bộ dữ liệu vào thư mục `data/` theo đúng cấu trúc đã mô tả trong `README.md` trước khi chạy các cell bên dưới.



## 1. Cài đặt Dependencies

Cell dưới đây sẽ cài đặt tất cả các thư viện cần thiết từ file `requirements.txt`. Nếu bạn đã cài đặt chúng trước đó, bạn có thể bỏ qua cell này.


In [ ]:
%pip install -r requirements.txt


## 2. Imports

Cell này tập hợp tất cả các thư viện sẽ được sử dụng trong toàn bộ notebook.


In [ ]:
# --- Python Standard Library ---
import math
import time
import json
import random
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional
from functools import partial
from multiprocessing import Pool, cpu_count

# --- Third-party Libraries ---
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from torch.optim.lr_scheduler import CosineAnnealingLR, StepLR

import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from PIL import Image
import timm
from einops import rearrange

# --- Visualization & Data Augmentation ---
import albumentations as A
from albumentations.pytorch import ToTensorV2

# --- Evaluation ---
try:
    from nltk.translate.bleu_score import corpus_bleu
except ImportError:
    print("Warning: NLTK not found. Using a fallback BLEU calculation.")
    # A simple fallback can be added here if needed, or just let it fail later.
    # For this notebook, we assume NLTK is installed via requirements.txt
    pass

print("All libraries imported successfully!")


## 3. Configuration

Cell này chứa lớp `Config` để quản lý tất cả các tham số của dự án. Bạn có thể thay đổi các giá trị tại đây để thử nghiệm.


In [ ]:
class Config:
    # Dataset paths
    DATA_DIR = Path("./data")
    TRAIN_IMAGE_DIR = DATA_DIR / "off_image_train"
    TEST_IMAGE_DIR = DATA_DIR / "off_image_test"
    TRAIN_CAPTION_FILE = DATA_DIR / "train_caption.txt"
    TEST_CAPTION_FILE = DATA_DIR / "test_caption.txt"
    DICTIONARY_FILE = DATA_DIR / "dictionary.txt"
    TRAIN_PROCESSED_IMAGE_DIR = DATA_DIR / "train_processed"
    TEST_PROCESSED_IMAGE_DIR = DATA_DIR / "test_processed"

    # Model parameters
    IMAGE_SIZE = 224
    PATCH_SIZE = 16
    EMBED_DIM = 256
    NUM_HEADS = 8
    NUM_LAYERS = 2
    MLP_RATIO = 4.0
    DROPOUT = 0.35
    ATTENTION_DROPOUT = 0.25
    APPLY_PREPROCESSING = True

    # Sequence parameters
    MAX_SEQ_LENGTH = 100
    PAD_TOKEN = "<PAD>"
    SOS_TOKEN = "<SOS>"
    EOS_TOKEN = "<EOS>"
    UNK_TOKEN = "<UNK>"

    # Training parameters
    BATCH_SIZE = 6
    ACCUMULATION_STEPS = 3
    LEARNING_RATE = 1.5e-4
    MIN_LR = 1e-6
    WEIGHT_DECAY = 2e-4
    NUM_EPOCHS = 60
    WARMUP_EPOCHS = 4
    GRADIENT_CLIP = 0.6

    # Mixed precision training
    USE_MIXED_PRECISION = True

    # Device
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Paths for saving
    CHECKPOINT_DIR = Path("checkpoints")
    LOG_DIR = Path("logs")
    OUTPUT_DIR = Path("outputs")

    # Create directories
    CHECKPOINT_DIR.mkdir(exist_ok=True)
    LOG_DIR.mkdir(exist_ok=True)
    OUTPUT_DIR.mkdir(exist_ok=True)

    # Data augmentation
    AUGMENT_PROB = 0.9
    ROTATION_RANGE = 12
    BRIGHTNESS_RANGE = 0.35
    CONTRAST_RANGE = 0.35
    
    # Regularization
    LABEL_SMOOTHING = 0.1
    
    # Evaluation
    EVAL_EVERY_N_EPOCHS = 5

    # Memory optimization
    GRADIENT_CHECKPOINTING = True
    PIN_MEMORY = True
    NUM_WORKERS = 2 

    def get_effective_batch_size(self):
        """Get effective batch size with gradient accumulation"""
        return self.BATCH_SIZE * self.ACCUMULATION_STEPS

    def print_config(self):
        """Print configuration details"""
        print("="*60)
        print("CONFIGURATION")
        print("="*60)
        print(f"Device: {self.DEVICE}")
        print(f"Image Size: {self.IMAGE_SIZE}x{self.IMAGE_SIZE}")
        print(f"Model Dimension: {self.EMBED_DIM}")
        print(f"Batch Size: {self.BATCH_SIZE} (Effective: {self.get_effective_batch_size()})")
        print(f"Learning Rate: {self.LEARNING_RATE}")
        print(f"Number of Epochs: {self.NUM_EPOCHS}")
        print(f"Mixed Precision: {self.USE_MIXED_PRECISION}")
        print(f"Gradient Checkpointing: {self.GRADIENT_CHECKPOINTING}")
        print("="*60)

# Instantiate config
config = Config()
config.print_config()


## 4. Utility Functions

Các hàm và lớp tiện ích hỗ trợ cho việc training, ví dụ như tính toán giá trị trung bình, lưu checkpoint, và tính BLEU score.


In [ ]:
class AverageMeter:
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()
    
    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0
    
    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

def calculate_bleu_score(predictions: List[str], references: List[List[str]]) -> float:
    """Calculate BLEU score for predictions"""
    try:
        pred_tokens = [pred.split() for pred in predictions]
        ref_tokens = [[ref[0].split()] for ref in references]
        
        bleu = corpus_bleu(ref_tokens, pred_tokens)
        return bleu
    except Exception as e:
        print(f"Could not calculate BLEU score: {e}")
        return 0.0

def save_checkpoint(state: Dict[str, Any], is_best: bool, checkpoint_dir: Path):
    """Save model checkpoint"""
    checkpoint_dir.mkdir(exist_ok=True)
    
    latest_path = checkpoint_dir / "latest.pth"
    torch.save(state, latest_path)
    
    if is_best:
        best_path = checkpoint_dir / "best.pth"
        torch.save(state, best_path)
        print(f"🎉 New best model saved with BLEU: {state.get('best_bleu', 0):.4f}")

def load_checkpoint(checkpoint_path: Path, model: nn.Module, optimizer=None, scheduler=None) -> Dict[str, Any]:
    """Load model checkpoint"""
    if not checkpoint_path.exists():
        print(f"Checkpoint not found: {checkpoint_path}, starting from scratch.")
        return {'epoch': 0, 'best_bleu': 0.0}
    
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    
    model.load_state_dict(checkpoint['model_state_dict'])
    
    if optimizer and 'optimizer_state_dict' in checkpoint:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    
    if scheduler and 'scheduler_state_dict' in checkpoint:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    
    print(f"Loaded checkpoint from epoch {checkpoint.get('epoch', 0)}")
    print(f"Best BLEU from checkpoint: {checkpoint.get('best_bleu', 0.0):.4f}")
    
    return checkpoint


## 5. Data Preprocessing

Đây là bước tiền xử lý dữ liệu. Các hàm dưới đây sẽ đọc ảnh từ các thư mục `off_image_*`, thực hiện các bước chuẩn hóa (binarize, resize, padding) và lưu kết quả vào các thư mục `*_processed`.

Việc chạy tiền xử lý một lần duy nhất giúp tăng tốc đáng kể quá trình huấn luyện sau này, vì `DataLoader` sẽ chỉ cần đọc các ảnh đã được xử lý sẵn.


In [ ]:
# =========================================================================================
# CÁC HÀM XỬ LÝ ẢNH
# =========================================================================================

def _preprocess_image_nb(image: np.ndarray) -> np.ndarray:
    """Chuyển ảnh sang dạng nhị phân (nền trắng, chữ đen)."""
    if len(image.shape) == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    else:
        gray = image
    
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    
    white_ratio = (binary > 127).mean()
    if white_ratio < 0.5:
        binary = 255 - binary
        
    return cv2.cvtColor(binary, cv2.COLOR_GRAY2RGB)

def _resize_with_padding_nb(image: np.ndarray, target_size: int) -> np.ndarray:
    """Resize ảnh về kích thước vuông, giữ nguyên tỉ lệ và thêm padding trắng."""
    h, w = image.shape[:2]
    if h == 0 or w == 0:
        return np.full((target_size, target_size, 3), 255, dtype=np.uint8)

    scale = target_size / max(h, w)
    new_w, new_h = int(w * scale), int(h * scale)
    
    interpolation = cv2.INTER_CUBIC if scale > 1 else cv2.INTER_AREA
    resized_image = cv2.resize(image, (new_w, new_h), interpolation=interpolation)
    
    final_image = np.full((target_size, target_size, 3), 255, dtype=np.uint8)
    pad_top = (target_size - new_h) // 2
    pad_left = (target_size - new_w) // 2
    final_image[pad_top:pad_top + new_h, pad_left:pad_left + new_w] = resized_image
    return final_image

def process_and_save_nb(image_info: tuple, config: Config, dest_dir: Path):
    """Hàm thực hiện toàn bộ pipeline cho một ảnh và lưu lại."""
    image_name, source_dir = image_info
    
    possible_extensions = ['.bmp', '.png', '.jpg', '.jpeg']
    found_path = None
    for ext in possible_extensions:
        p = (source_dir / image_name).with_suffix(ext)
        if p.exists():
            found_path = p
            break
            
    if not found_path:
        return

    try:
        image = cv2.imread(str(found_path))
        if image is None: return
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        if config.APPLY_PREPROCESSING:
            image = _preprocess_image_nb(image)
        
        image = _resize_with_padding_nb(image, config.IMAGE_SIZE)
        
        output_path = (dest_dir / image_name).with_suffix('.png')
        cv2.imwrite(str(output_path), cv2.cvtColor(image, cv2.COLOR_RGB2BGR))

    except Exception as e:
        print(f"Lỗi khi xử lý ảnh {found_path}: {e}")

def get_image_list_from_caption_nb(caption_file: Path, source_dir: Path) -> list:
    """Đọc danh sách tên ảnh từ file caption."""
    image_infos = []
    with open(caption_file, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 1:
                image_name = parts[0]
                image_infos.append((image_name, source_dir))
    return image_infos

# =========================================================================================
# HÀM CHẠY TIỀN XỬ LÝ
# =========================================================================================

def run_preprocessing(config: Config):
    """Hàm chính để chạy toàn bộ quá trình tiền xử lý."""
    start_time = time.time()
    
    datasets_to_process = {
        "train": (config.TRAIN_IMAGE_DIR, config.TRAIN_CAPTION_FILE, config.TRAIN_PROCESSED_IMAGE_DIR),
        "test": (config.TEST_IMAGE_DIR, config.TEST_CAPTION_FILE, config.TEST_PROCESSED_IMAGE_DIR)
    }

    for name, (source_dir, caption_file, dest_dir) in datasets_to_process.items():
        print(f"\nBắt đầu xử lý bộ dữ liệu: {name.upper()}")
        print(f"Thư mục nguồn: {source_dir}")
        print(f"Thư mục đích:  {dest_dir}")
        
        dest_dir.mkdir(parents=True, exist_ok=True)
        
        image_infos = get_image_list_from_caption_nb(caption_file, source_dir)
        print(f"Tìm thấy {len(image_infos)} ảnh để xử lý.")

        num_processes = max(1, cpu_count() - 1)
        print(f"Sử dụng {num_processes} nhân CPU để xử lý song song...")
        
        processor = partial(process_and_save_nb, config=config, dest_dir=dest_dir)
        
        with Pool(processes=num_processes) as pool:
            list(tqdm(pool.imap_unordered(processor, image_infos), total=len(image_infos)))

    end_time = time.time()
    print(f"\nHoàn tất toàn bộ quá trình tiền xử lý trong {end_time - start_time:.2f} giây.")


## 6. Dataset and DataLoader

Các lớp và hàm trong cell này định nghĩa cách nạp dữ liệu đã được tiền xử lý.

-   `MathExpressionDataset`: Đọc ảnh từ thư mục `*_processed` và thực hiện data augmentation on-the-fly.
-   `create_dataloaders`: Tạo ra các đối tượng `DataLoader` cho tập training và testing.


In [ ]:
class MathExpressionDataset(Dataset):
    """
    Dataset đọc trực tiếp các ảnh ĐÃ ĐƯỢC TIỀN XỬ LÝ.
    Nó chỉ thực hiện các tác vụ nhẹ on-the-fly như Data Augmentation.
    """
    def __init__(self, processed_image_dir: Path, caption_file: Path, dictionary_file: Path, config: Config, is_train: bool = True):
        self.image_dir = Path(processed_image_dir)
        self.is_train = is_train
        self.config = config

        self.vocab = self._load_vocabulary(dictionary_file)
        self.idx_to_token = {v: k for k, v in self.vocab.items()}
        self.data = self._load_captions(caption_file)
        
        self.transform = self._get_transform()

    def _load_vocabulary(self, dictionary_file: Path) -> Dict[str, int]:
        vocab = {self.config.PAD_TOKEN: 0, self.config.SOS_TOKEN: 1, self.config.EOS_TOKEN: 2, self.config.UNK_TOKEN: 3}
        with open(dictionary_file, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if '\t' in line:
                    token, idx = line.split('\t')
                    vocab[token] = int(idx) + 4
        return vocab

    def _load_captions(self, caption_file: Path) -> List[Tuple[str, str]]:
        data = []
        with open(caption_file, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) >= 2:
                    data.append((parts[0], '\t'.join(parts[1:])))
        return data

    def _get_transform(self) -> A.Compose:
        normalize = A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        if self.is_train:
            return A.Compose([
                A.Rotate(limit=self.config.ROTATION_RANGE, p=self.config.AUGMENT_PROB, border_mode=cv2.BORDER_CONSTANT, value=[255, 255, 255]),
                A.RandomBrightnessContrast(brightness_limit=self.config.BRIGHTNESS_RANGE,
                                           contrast_limit=self.config.CONTRAST_RANGE, p=self.config.AUGMENT_PROB),
                normalize,
                ToTensorV2()
            ])
        else:
            return A.Compose([normalize, ToTensorV2()])

    def _tokenize_caption(self, caption: str) -> List[int]:
        tokens = caption.split()
        indices = [self.vocab[self.config.SOS_TOKEN]] + [self.vocab.get(t, self.vocab[self.config.UNK_TOKEN]) for t in tokens] + [self.vocab[self.config.EOS_TOKEN]]
        return indices

    def _pad_sequence(self, sequence: List[int], max_length: int) -> List[int]:
        if len(sequence) >= max_length: return sequence[:max_length]
        return sequence + [self.vocab[self.config.PAD_TOKEN]] * (max_length - len(sequence))

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int) -> Dict:
        image_name, caption = self.data[idx]
        image_path = self.image_dir / f"{image_name}.png"

        try:
            image = cv2.imread(str(image_path))
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        except (IOError, cv2.error):
            image = np.ones((self.config.IMAGE_SIZE, self.config.IMAGE_SIZE, 3), dtype=np.uint8) * 255

        transformed = self.transform(image=image)
        image_tensor = transformed['image']
        
        token_indices = self._tokenize_caption(caption)
        input_seq = self._pad_sequence(token_indices[:-1], self.config.MAX_SEQ_LENGTH)
        target_seq = self._pad_sequence(token_indices[1:], self.config.MAX_SEQ_LENGTH)

        return {
            'image': image_tensor,
            'input_seq': torch.tensor(input_seq, dtype=torch.long),
            'target_seq': torch.tensor(target_seq, dtype=torch.long),
            'caption': caption,
            'image_name': image_name
        }

def _pad_collate_fn_nb(batch):
    images = [item['image'] for item in batch]
    images_tensor = torch.stack(images, dim=0)
    
    return {
        'image': images_tensor,
        'input_seq': torch.stack([item['input_seq'] for item in batch]),
        'target_seq': torch.stack([item['target_seq'] for item in batch]),
        'caption': [item['caption'] for item in batch],
        'image_name': [item['image_name'] for item in batch]
    }


def create_dataloaders_nb(config: Config) -> Tuple[DataLoader, DataLoader]:
    print("Tạo dataloader từ dữ liệu ĐÃ TIỀN XỬ LÝ.")

    train_dataset = MathExpressionDataset(
        processed_image_dir=config.TRAIN_PROCESSED_IMAGE_DIR,
        caption_file=config.TRAIN_CAPTION_FILE,
        dictionary_file=config.DICTIONARY_FILE,
        config=config,
        is_train=True
    )
    test_dataset = MathExpressionDataset(
        processed_image_dir=config.TEST_PROCESSED_IMAGE_DIR,
        caption_file=config.TEST_CAPTION_FILE,
        dictionary_file=config.DICTIONARY_FILE,
        config=config,
        is_train=False
    )

    train_loader = DataLoader(
        train_dataset, batch_size=config.BATCH_SIZE, shuffle=True,
        num_workers=config.NUM_WORKERS, pin_memory=config.PIN_MEMORY, drop_last=True,
        collate_fn=_pad_collate_fn_nb
    )
    test_loader = DataLoader(
        test_dataset, batch_size=config.BATCH_SIZE, shuffle=False,
        num_workers=config.NUM_WORKERS, pin_memory=config.PIN_MEMORY,
        collate_fn=_pad_collate_fn_nb
    )

    return train_loader, test_loader


## 7. Model Architecture

Cell này chứa toàn bộ định nghĩa kiến trúc của mô hình `MathExpressionVisionTransformer`, bao gồm cả encoder và decoder.
- **Vision Encoder**: Sử dụng `timm` để tải mô hình `vit-small` đã được huấn luyện trước.
- **Text Decoder**: Một Transformer Decoder chuẩn để sinh ra chuỗi token LaTeX.


In [ ]:
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, decoder_dim: int, encoder_dim: int, num_heads: int, dropout: float = 0.1, attn_dropout: float = 0.1):
        super().__init__()
        assert decoder_dim % num_heads == 0, "Decoder dimension must be divisible by num_heads"
        self.embed_dim = decoder_dim
        self.num_heads = num_heads
        self.head_dim = self.embed_dim // num_heads
        self.scale = self.head_dim ** -0.5

        self.q = nn.Linear(decoder_dim, self.embed_dim, bias=False)
        self.kv = nn.Linear(encoder_dim, self.embed_dim * 2, bias=False)
        self.attn_drop = nn.Dropout(attn_dropout)
        self.proj = nn.Linear(self.embed_dim, self.embed_dim)
        self.proj_drop = nn.Dropout(dropout)

    def forward(self, query: torch.Tensor, key_value: torch.Tensor) -> torch.Tensor:
        B, N_q, C_q = query.shape
        B, N_kv, C_kv = key_value.shape

        q = self.q(query).reshape(B, N_q, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        
        kv = self.kv(key_value).reshape(B, N_kv, 2, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        k, v = kv[0], kv[1]

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = F.softmax(attn, dim=-1)
        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B, N_q, self.embed_dim)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x

class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int, dropout: float = 0.1, attn_dropout: float = 0.1):
        super().__init__()
        assert embed_dim % num_heads == 0, "Embedding dimension must be divisible by num_heads"
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5

        self.qkv = nn.Linear(embed_dim, embed_dim * 3, bias=False)
        self.attn_drop = nn.Dropout(attn_dropout)
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.proj_drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        attn = (q @ k.transpose(-2, -1)) * self.scale

        if mask is not None:
            attn = attn.masked_fill(mask == 0, -1e9)

        attn = F.softmax(attn, dim=-1)
        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x
        
class MLP(nn.Module):
    def __init__(self, embed_dim: int, mlp_ratio: float = 4.0, dropout: float = 0.1):
        super().__init__()
        self.fc1 = nn.Linear(embed_dim, int(embed_dim * mlp_ratio))
        self.act = nn.GELU()
        self.fc2 = nn.Linear(int(embed_dim * mlp_ratio), embed_dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x

class TransformerDecoderBlock(nn.Module):
    def __init__(self, embed_dim: int, encoder_embed_dim: int, num_heads: int, mlp_ratio: float = 4.0, dropout: float = 0.1, attn_dropout: float = 0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.self_attn = MultiHeadAttention(embed_dim, num_heads, dropout, attn_dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.cross_attn = MultiHeadCrossAttention(embed_dim, encoder_embed_dim, num_heads, dropout, attn_dropout)
        self.norm3 = nn.LayerNorm(embed_dim)
        self.mlp = MLP(embed_dim, mlp_ratio, dropout)
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, encoder_output: torch.Tensor, tgt_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        x = x + self.drop(self.self_attn(self.norm1(x), tgt_mask))
        x = x + self.drop(self.cross_attn(self.norm2(x), encoder_output))
        x = x + self.drop(self.mlp(self.norm3(x)))
        return x

class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, encoder_embed_dim: int, num_layers: int, num_heads: int, mlp_ratio: float, dropout: float, attn_dropout: float, max_seq_length: int):
        super().__init__()
        self.embed_dim = embed_dim
        self.token_embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_embed = nn.Parameter(torch.zeros(1, max_seq_length, embed_dim))
        self.dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            TransformerDecoderBlock(embed_dim, encoder_embed_dim, num_heads, mlp_ratio, dropout, attn_dropout) for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, vocab_size)
        
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def create_causal_mask(self, seq_len: int, device) -> torch.Tensor:
        return torch.triu(torch.full((seq_len, seq_len), float('-inf'), device=device), diagonal=1)

    def forward(self, tgt: torch.Tensor, encoder_output: torch.Tensor, tgt_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        B, seq_len = tgt.shape
        x = self.token_embed(tgt) * math.sqrt(self.embed_dim)
        x = x + self.pos_embed[:, :seq_len, :]
        x = self.dropout(x)

        if tgt_mask is None:
            tgt_mask = self.create_causal_mask(seq_len, tgt.device)

        for block in self.blocks:
            x = block(x, encoder_output, tgt_mask)

        x = self.norm(x)
        return self.head(x)

class MathExpressionVisionTransformer(nn.Module):
    def __init__(self, config: Config, vocab_size: int):
        super().__init__()
        self.config = config
        self.vision_encoder = timm.create_model(
            'vit_small_patch16_224.augreg_in21k',
            pretrained=True,
            num_classes=0, 
            img_size=config.IMAGE_SIZE,
        )
        
        encoder_embed_dim = self.vision_encoder.embed_dim
        
        # Potentially freeze the encoder
        # for param in self.vision_encoder.parameters():
        #     param.requires_grad = False
            
        self.encoder_projection = nn.Linear(encoder_embed_dim, config.EMBED_DIM)

        self.text_decoder = TransformerDecoder(
            vocab_size=vocab_size,
            embed_dim=config.EMBED_DIM,
            encoder_embed_dim=config.EMBED_DIM,
            num_layers=max(config.NUM_LAYERS // 2, 2), # Decoder has fewer layers
            num_heads=config.NUM_HEADS,
            mlp_ratio=config.MLP_RATIO,
            dropout=config.DROPOUT,
            attn_dropout=config.ATTENTION_DROPOUT,
            max_seq_length=config.MAX_SEQ_LENGTH
        )
        if config.GRADIENT_CHECKPOINTING:
            self.vision_encoder.set_grad_checkpointing()


    def forward(self, images: torch.Tensor, input_seq: torch.Tensor) -> torch.Tensor:
        encoder_output = self.vision_encoder.forward_features(images)
        encoder_output = self.encoder_projection(encoder_output)
        logits = self.text_decoder(input_seq, encoder_output)
        return logits

    def generate(self, images: torch.Tensor, sos_token_id: int, eos_token_id: int, max_length: int) -> torch.Tensor:
        batch_size = images.size(0)
        device = images.device
        
        encoder_output = self.vision_encoder.forward_features(images)
        encoder_output = self.encoder_projection(encoder_output)
        
        generated = torch.full((batch_size, 1), sos_token_id, device=device, dtype=torch.long)

        for _ in range(max_length - 1):
            logits = self.text_decoder(generated, encoder_output)
            next_token_logits = logits[:, -1, :]
            next_token = torch.argmax(next_token_logits, dim=-1).unsqueeze(1)
            generated = torch.cat([generated, next_token], dim=1)
            if (next_token == eos_token_id).all():
                break
                
        return generated
        
print("Model architecture is defined.")


## 8. Training Logic

Đây là nơi chứa logic chính cho việc huấn luyện và đánh giá mô hình.
-   `Trainer` class: Đóng gói toàn bộ quy trình, bao gồm vòng lặp training, vòng lặp validation, tính toán metrics, và lưu lại checkpoints.


In [ ]:
class Trainer:
    def __init__(self, config: Config):
        self.config = config
        self.device = config.DEVICE
        self.train_loader, self.test_loader = create_dataloaders_nb(config)

        self.vocab_size = len(self.train_loader.dataset.vocab)
        self.vocab = self.train_loader.dataset.vocab
        self.idx_to_token = self.train_loader.dataset.idx_to_token

        self.model = MathExpressionVisionTransformer(config, self.vocab_size).to(self.device)

        self.criterion = nn.CrossEntropyLoss(
            ignore_index=self.vocab[config.PAD_TOKEN],
            label_smoothing=config.LABEL_SMOOTHING
        )
        self.optimizer = optim.AdamW(
            self.model.parameters(),
            lr=config.LEARNING_RATE,
            weight_decay=config.WEIGHT_DECAY
        )
        self.scheduler = CosineAnnealingLR(
            self.optimizer,
            T_max=config.NUM_EPOCHS,
            eta_min=config.MIN_LR
        )

        self.writer = SummaryWriter(config.LOG_DIR / f"run_notebook_{int(time.time())}")
        self.epoch = 0
        self.best_bleu = 0.0

        print(f"Model created with {sum(p.numel() for p in self.model.parameters()):,} parameters")
        print(f"Training on {self.device}")

    def train_epoch(self) -> Dict[str, float]:
        self.model.train()
        losses, accuracies = AverageMeter(), AverageMeter()
        scaler = torch.amp.GradScaler(enabled=self.config.USE_MIXED_PRECISION)
        
        pbar = tqdm(self.train_loader, desc=f"Epoch {self.epoch+1}/{self.config.NUM_EPOCHS} [Training]")

        for batch_idx, batch in enumerate(pbar):
            images = batch['image'].to(self.device)
            input_seq = batch['input_seq'].to(self.device)
            target_seq = batch['target_seq'].to(self.device)

            with torch.amp.autocast(device_type='cuda', dtype=torch.float16, enabled=self.config.USE_MIXED_PRECISION):
                logits = self.model(images, input_seq)
                loss = self.criterion(logits.view(-1, self.vocab_size), target_seq.view(-1))

            if not torch.isfinite(loss): continue

            scaler.scale(loss / self.config.ACCUMULATION_STEPS).backward()

            if (batch_idx + 1) % self.config.ACCUMULATION_STEPS == 0:
                scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config.GRADIENT_CLIP)
                scaler.step(self.optimizer)
                scaler.update()
                self.optimizer.zero_grad()
            
            mask = target_seq != self.vocab[self.config.PAD_TOKEN]
            predictions = torch.argmax(logits, dim=-1)
            accuracy = ((predictions == target_seq) & mask).sum().float() / mask.sum().float()

            losses.update(loss.item(), images.size(0))
            accuracies.update(accuracy.item(), images.size(0))
            pbar.set_postfix({'Loss': f'{losses.avg:.4f}', 'Acc': f'{accuracies.avg:.4f}'})

        return {'loss': losses.avg, 'accuracy': accuracies.avg}

    def evaluate(self) -> Dict[str, float]:
        self.model.eval()
        losses, accuracies = AverageMeter(), AverageMeter()
        all_predictions, all_targets = [], []

        with torch.no_grad():
            pbar = tqdm(self.test_loader, desc="[Evaluating]")
            for batch in pbar:
                images = batch['image'].to(self.device)
                target_seq = batch['target_seq'].to(self.device)

                generated = self.model.generate(
                    images, self.vocab[self.config.SOS_TOKEN], self.vocab[self.config.EOS_TOKEN],
                    max_length=self.config.MAX_SEQ_LENGTH
                )

                for i in range(generated.size(0)):
                    pred_str = self._indices_to_tokens(generated[i].cpu().numpy())
                    tgt_str = self._indices_to_tokens(target_seq[i].cpu().numpy())
                    all_predictions.append(pred_str)
                    all_targets.append([tgt_str])
        
        bleu = calculate_bleu_score(all_predictions, all_targets)
        # In pure evaluation, loss/accuracy on logits is less critical than generated BLEU
        return {'bleu': bleu}

    def _indices_to_tokens(self, indices: np.ndarray) -> str:
        tokens = []
        for idx in indices:
            if idx == self.vocab[self.config.EOS_TOKEN]: break
            if idx not in [self.vocab[self.config.PAD_TOKEN], self.vocab[self.config.SOS_TOKEN]]:
                tokens.append(self.idx_to_token.get(idx, self.config.UNK_TOKEN))
        return ' '.join(tokens)

    def train(self):
        print(f"\nStarting training for {self.config.NUM_EPOCHS} epochs...")
        for epoch in range(self.config.NUM_EPOCHS):
            self.epoch = epoch
            train_metrics = self.train_epoch()
            self.scheduler.step()
            
            self.writer.add_scalar("Epoch/Train_Loss", train_metrics['loss'], epoch)
            self.writer.add_scalar("Epoch/Train_Accuracy", train_metrics['accuracy'], epoch)
            self.writer.add_scalar("Epoch/LR", self.optimizer.param_groups[0]['lr'], epoch)

            print(f"\nEpoch {epoch+1}/{self.config.NUM_EPOCHS} Summary:\n"
                  f"  Train Loss: {train_metrics['loss']:.4f}, Train Acc: {train_metrics['accuracy']:.4f}")

            if (epoch + 1) % self.config.EVAL_EVERY_N_EPOCHS == 0:
                eval_metrics = self.evaluate()
                self.writer.add_scalar("Epoch/Val_BLEU", eval_metrics['bleu'], epoch)
                print(f"  Validation BLEU: {eval_metrics['bleu']:.4f}")

                is_best = eval_metrics['bleu'] > self.best_bleu
                if is_best: self.best_bleu = eval_metrics['bleu']

                save_checkpoint({
                    'epoch': epoch + 1,
                    'model_state_dict': self.model.state_dict(),
                    'optimizer_state_dict': self.optimizer.state_dict(),
                    'scheduler_state_dict': self.scheduler.state_dict(),
                    'best_bleu': self.best_bleu,
                    'vocab': self.vocab,
                }, is_best, self.config.CHECKPOINT_DIR)
            else:
                 # Save latest checkpoint without running evaluation
                 save_checkpoint({
                    'epoch': epoch + 1,
                    'model_state_dict': self.model.state_dict(),
                    'optimizer_state_dict': self.optimizer.state_dict(),
                    'scheduler_state_dict': self.scheduler.state_dict(),
                    'best_bleu': self.best_bleu,
                    'vocab': self.vocab,
                }, False, self.config.CHECKPOINT_DIR)

        self.writer.close()
        print(f"\nTraining complete! Best BLEU: {self.best_bleu:.4f}")


## 9. Execute Workflow

Bây giờ, chúng ta sẽ thực thi các bước đã định nghĩa ở trên.

### 9.1. Run Preprocessing

Chạy cell dưới đây để bắt đầu quá trình tiền xử lý. Quá trình này chỉ cần chạy **một lần duy nhất**. Nếu bạn đã chạy nó trước đó và các thư mục `*_processed` đã tồn tại, bạn có thể bỏ qua cell này.


In [ ]:
# Tạo một đối tượng config mới để chắc chắn các đường dẫn là chính xác
preprocessing_config = Config()
run_preprocessing(preprocessing_config)


### 9.2. Run Training

Sau khi dữ liệu đã được tiền xử lý, chạy cell này để bắt đầu huấn luyện mô hình.


In [ ]:
# Tạo đối tượng config và trainer mới để bắt đầu huấn luyện
training_config = Config()
trainer = Trainer(training_config)
trainer.train()
